# Phase 4 — Model explainability (winner ensemble)

This notebook is **interpretability only** — it does not retrain models.

It loads the Phase 2 winner from `results_phase2.zip` and explains **what the model uses to make its anomaly score predictable** on labelled `dev_test`.

Anomaly score in this project is **mean squared reconstruction error (MSE)** averaged over mel bins and time. Explainability therefore focuses on:

- **Where** reconstruction fails (per-pixel squared error maps)
- **Which mel frequency bins** carry most error (mel-bin curves)
- **Whether bottleneck embeddings** separate labels / domains / sections (PCA of pooled `z`)
- **Input attribution** via Integrated Gradients of the scalar MSE w.r.t. the log-mel input

> Note: `adversarial` training adds a domain loss on `z` during training, but **inference scoring is reconstruction MSE only**. This notebook explains the MSE signal; it does not claim the domain head is used at inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/Unsupervised_anomalous_sound

import os, json, glob, zipfile, shutil
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from IPython.display import Markdown, display, Image
from sklearn.decomposition import PCA

from model import build_model
from dataset import BearingDataset

In [ ]:
# ---- knobs ----
RESULTS_ZIP = "results_phase2.zip"
EXTRACT_DIR = "/content/results_phase2"
OUT_DIR = "/content/phase4_outputs"
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

DEV_TEST_DIR = "data/dcase_bearing_dev/bearing/test"
DEV_TEST_CACHE = "/content/bearing_cache_dev_test"

BATCH_SIZE = 32
N_EXAMPLES_HIGH = 6   # high-MSE anomalies
N_EXAMPLES_LOW = 6    # low-MSE normals
IG_STEPS = 32         # integrated-gradients steps (higher = smoother, slower)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# ---- load winner checkpoints ----
with zipfile.ZipFile(RESULTS_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)

winner_path = os.path.join(EXTRACT_DIR, "best_model", "WINNER.json")
with open(winner_path) as f:
    winner = json.load(f)

key = winner["key"]
arch = winner["arch"]
ckpts = sorted(glob.glob(os.path.join(EXTRACT_DIR, "best_model", f"{key}_fold*.pt")))
if not ckpts:
    raise FileNotFoundError(f"No checkpoints for {key} under {EXTRACT_DIR}/best_model")

models = []
for p in ckpts:
    ckpt = torch.load(p, map_location=device, weights_only=False)
    m = build_model(arch).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    models.append(m)

display(Markdown(f"### Winner\n- `{winner['mode']}/{winner['arch']}`\n- folds: {len(ckpts)}\n- key: `{key}`"))

In [ ]:
# ---- dataset + ensemble MSE vector ----
dev_ds = BearingDataset(DEV_TEST_DIR, cache_dir=DEV_TEST_CACHE, return_label=True, verbose=False)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                        pin_memory=(device == "cuda"))

def ensemble_forward(x):
    """Return ensemble-mean recon and per-fold recons. x: (B,1,H,T)"""
    recons = []
    for m in models:
        recon, _ = m(x)
        recons.append(recon)
    # align time
    t = min(r.shape[-1] for r in recons + [x])
    x0 = x[..., :t]
    recons_t = [r[..., :t] for r in recons]
    recon_mean = torch.stack(recons_t, dim=0).mean(dim=0)
    return recon_mean, recons_t, x0

@torch.no_grad()
def per_clip_mse(x):
    recon_mean, _, x0 = ensemble_forward(x)
    return ((recon_mean - x0) ** 2).mean(dim=(1, 2, 3))

all_scores = []
all_dom, all_sec, all_lab, all_name = [], [], [], []
with torch.no_grad():
    for x, dom, sec, lab in dev_loader:
        x = x.to(device, non_blocking=True)
        s = per_clip_mse(x).detach().cpu().numpy()
        all_scores.append(s)
        all_dom.extend(list(dom))
        all_sec.extend([int(v) for v in sec])
        all_lab.extend(list(lab))

scores = np.concatenate(all_scores)
domains = np.array(all_dom)
sections = np.array(all_sec)
labels = np.array(all_lab)
names = np.array([dev_ds.samples[i]["filename"] for i in range(len(dev_ds))])

display(Markdown(f"### dev_test scored\n- clips: {len(scores)}"))


In [ ]:
# ---- A) mel-bin mean squared error curves (label × section × domain pooled variants) ----
explain_dir = os.path.join(OUT_DIR, "explain")
os.makedirs(explain_dir, exist_ok=True)

mel_sum = {}
mel_cnt = {}

def _key(lab, sec=None, dom=None):
    if sec is None and dom is None:
        return f"label={lab}"
    if sec is None:
        return f"label={lab}|dom={dom}"
    if dom is None:
        return f"label={lab}|sec={sec}"
    return f"label={lab}|sec={sec}|dom={dom}"

with torch.no_grad():
    idx = 0
    for x, dom, sec, lab in dev_loader:
        x = x.to(device, non_blocking=True)
        recon_mean, _, x0 = ensemble_forward(x)
        e = (recon_mean - x0) ** 2  # (B,1,H,T)
        mel = e.mean(dim=(1, 3)).squeeze(1).detach().cpu().numpy()  # (B,H)
        for i in range(x.size(0)):
            l = lab[i]
            s = int(sec[i])
            d = dom[i]
            for k in [
                _key(l),
                _key(l, sec=s),
                _key(l, dom=d),
                _key(l, sec=s, dom=d),
            ]:
                mel_sum[k] = mel_sum.get(k, None)
                if mel_sum[k] is None:
                    mel_sum[k] = mel[i].astype(np.float64)
                else:
                    mel_sum[k] += mel[i]
                mel_cnt[k] = mel_cnt.get(k, 0) + 1
        idx += x.size(0)

# pooled normal vs anomaly
def _curve(k):
    return mel_sum[k] / max(mel_cnt.get(k, 0), 1)

k_n, k_a = _key("normal"), _key("anomaly")
bins = np.arange(_curve(k_n).shape[0])
plt.figure(figsize=(9, 4.6))
plt.plot(bins, _curve(k_n), label="normal")
plt.plot(bins, _curve(k_a), label="anomaly")
plt.plot(bins, _curve(k_a) - _curve(k_n), color="k", alpha=0.6, label="anomaly - normal")
plt.title("Mel-bin mean squared reconstruction error (domain+section pooled)")
plt.xlabel("mel bin")
plt.ylabel("mean squared error")
plt.grid(alpha=0.3)
plt.legend()
p0 = os.path.join(explain_dir, "melbin_pooled_label.png")
plt.tight_layout(); plt.savefig(p0, dpi=140); plt.close()

# per section (00-02)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, sec in zip(axes, [0, 1, 2]):
    kn, ka = _key("normal", sec=sec), _key("anomaly", sec=sec)
    ax.plot(bins, _curve(kn), label="normal")
    ax.plot(bins, _curve(ka), label="anomaly")
    ax.plot(bins, _curve(ka) - _curve(kn), color="k", alpha=0.45, label="diff")
    ax.set_title(f"section {sec:02d}")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("mean squared error")
axes[-1].legend(loc="upper right")
p1 = os.path.join(explain_dir, "melbin_by_section.png")
plt.tight_layout(); plt.savefig(p1, dpi=140); plt.close()

display(Markdown("### Mel-bin attribution"))
display(Image(p0)); display(Image(p1))

In [ ]:
# ---- B) PCA of pooled bottleneck z (ensemble mean) ----
Z, y, dlist, slist = [], [], [], []
with torch.no_grad():
    for x, dom, sec, lab in dev_loader:
        x = x.to(device, non_blocking=True)
        zs = []
        for m in models:
            _, z = m(x)
            zp = z.mean(dim=[2, 3]).detach().cpu().numpy()
            zs.append(zp)
        z_ens = np.mean(np.stack(zs, axis=0), axis=0)
        for i in range(x.size(0)):
            Z.append(z_ens[i])
            y.append(1 if lab[i] == "anomaly" else 0)
            dlist.append(dom[i])
            slist.append(int(sec[i]))

Z = np.stack(Z, axis=0)
y = np.array(y)
darr = np.array(dlist)
sarr = np.array(slist)

Z2 = PCA(n_components=2, random_state=0).fit_transform(Z)

def _scatter(mask, title, path, legend=True):
    plt.figure(figsize=(6.2, 5.2))
    plt.scatter(Z2[mask, 0], Z2[mask, 1], s=16, alpha=0.55)
    plt.title(title)
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.grid(alpha=0.3)
    if legend:
        plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=140); plt.close()

plt.figure(figsize=(6.2, 5.2))
plt.scatter(Z2[y == 0, 0], Z2[y == 0, 1], s=16, alpha=0.55, label="normal")
plt.scatter(Z2[y == 1, 0], Z2[y == 1, 1], s=16, alpha=0.55, label="anomaly")
plt.title("PCA of pooled z (colored by label)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(alpha=0.3); plt.legend()
p_pca_lab = os.path.join(explain_dir, "pca_z_label.png")
plt.tight_layout(); plt.savefig(p_pca_lab, dpi=140); plt.close()

plt.figure(figsize=(6.2, 5.2))
for d in ["source", "target"]:
    m = darr == d
    plt.scatter(Z2[m, 0], Z2[m, 1], s=16, alpha=0.55, label=d)
plt.title("PCA of pooled z (colored by domain)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(alpha=0.3); plt.legend()
p_pca_dom = os.path.join(explain_dir, "pca_z_domain.png")
plt.tight_layout(); plt.savefig(p_pca_dom, dpi=140); plt.close()

plt.figure(figsize=(6.2, 5.2))
for s in sorted(set(sarr.tolist())):
    m = sarr == s
    plt.scatter(Z2[m, 0], Z2[m, 1], s=16, alpha=0.55, label=f"sec {s}")
plt.title("PCA of pooled z (colored by section)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(alpha=0.3); plt.legend()
p_pca_sec = os.path.join(explain_dir, "pca_z_section.png")
plt.tight_layout(); plt.savefig(p_pca_sec, dpi=140); plt.close()

display(Markdown("### Latent PCA"))
display(Image(p_pca_lab)); display(Image(p_pca_dom)); display(Image(p_pca_sec))

In [ ]:
# ---- C) Integrated Gradients of scalar MSE w.r.t. input log-mel ----
def integrated_gradients_mse(x0, steps=32):
    """Integrated gradients of scalar MSE(recon, x) w.r.t. input x.

    x0 must be detached from any previous graph (we clone internally).
    """
    x_ref = x0.detach().clone()
    baseline = torch.zeros_like(x_ref)
    grads = []
    for k in range(1, steps + 1):
        alpha = float(k) / float(steps)
        x = baseline + alpha * (x_ref - baseline)
        x = x.detach().requires_grad_(True)

        recon_mean, _, xt = ensemble_forward(x)
        t = min(recon_mean.shape[-1], xt.shape[-1])
        mse = ((recon_mean[..., :t] - xt[..., :t]) ** 2).mean()
        mse.backward()
        grads.append(x.grad.detach())

        for m in models:
            m.zero_grad(set_to_none=True)

    avg_grad = torch.stack(grads, dim=0).mean(dim=0)
    ig = (x_ref - baseline) * avg_grad
    return ig.squeeze(0).squeeze(0).detach().cpu().numpy()  # (H,T)

def save_triplet(inp, recon, err, ig, title, path):
    inp = np.squeeze(inp); recon = np.squeeze(recon); err = np.squeeze(err); ig = np.squeeze(ig)
    fig, ax = plt.subplots(1, 4, figsize=(16.5, 3.4))
    ax[0].imshow(inp, aspect="auto", origin="lower"); ax[0].set_title("input")
    ax[1].imshow(recon, aspect="auto", origin="lower"); ax[1].set_title("recon (ens.)")
    vmax = float(np.percentile(err, 99.0))
    im2 = ax[2].imshow(err, aspect="auto", origin="lower", vmin=0.0, vmax=max(vmax, 1e-8))
    ax[2].set_title("squared error")
    vmax3 = float(np.percentile(np.abs(ig), 99.0))
    im3 = ax[3].imshow(ig, aspect="auto", origin="lower", cmap="coolwarm",
                     vmin=-vmax3, vmax=vmax3)
    ax[3].set_title("IG(MSE) wrt input")
    for a in ax:
        a.set_xlabel("time"); a.set_ylabel("mel")
    fig.suptitle(title)
    fig.colorbar(im2, ax=ax[2], fraction=0.046, pad=0.04)
    fig.colorbar(im3, ax=ax[3], fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(path, dpi=170); plt.close(fig)

order = np.argsort(-scores)
anom_idx = [i for i in order.tolist() if labels[i] == "anomaly"][:N_EXAMPLES_HIGH]
norm_idx = [i for i in np.argsort(scores).tolist() if labels[i] == "normal"][:N_EXAMPLES_LOW]

ig_paths = []
for i in anom_idx + norm_idx:
    tag = "anom" if labels[i] == "anomaly" else "norm"
    x = dev_ds[i][0].unsqueeze(0).to(device)
    x0 = x.detach()

    recon_mean, _, x0a = ensemble_forward(x0)
    t = min(recon_mean.shape[-1], x0a.shape[-1])
    inp = x0a[..., :t].squeeze(0).squeeze(0).detach().cpu().numpy()
    rec = recon_mean[..., :t].squeeze(0).squeeze(0).detach().cpu().numpy()
    err = (recon_mean[..., :t] - x0a[..., :t]).pow(2).squeeze(0).squeeze(0).detach().cpu().numpy()

    ig = integrated_gradients_mse(x0, steps=IG_STEPS)
    ig = ig[:, :t]

    title = f"{tag} | {names[i]} | score={float(scores[i]):.6f}"
    outp = os.path.join(explain_dir, f"ig_{tag}_{i}.png")
    save_triplet(inp, rec, err, ig, title, outp)
    ig_paths.append(outp)

display(Markdown("### Integrated gradients examples"))
for pth in ig_paths:
    display(Image(pth))


In [ ]:
# ---- D) bundle outputs ----
summary = {
    "winner": winner,
    "outputs": {
        "explain_dir": explain_dir,
        "melbin_pooled": p0,
        "melbin_by_section": p1,
        "pca_label": p_pca_lab,
        "pca_domain": p_pca_dom,
        "pca_section": p_pca_sec,
        "integrated_gradients": ig_paths,
    },
}
with open(os.path.join(OUT_DIR, "phase4_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

zip_path = shutil.make_archive("/content/phase4_outputs", "zip", OUT_DIR)
print("zip:", zip_path)

drive_zip = os.path.join(os.getcwd(), "phase4_outputs.zip")
if os.path.exists(drive_zip):
    os.remove(drive_zip)
shutil.copy2(zip_path, drive_zip)
print("copied to:", drive_zip)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("download skipped:", e)